# Note 01: First CUDA Program

CUDA C++ programs have a host side that runs on the CPU and a device side that runs on the GPU. This note builds a minimal GPU kernel and uses it to establish that separation.

## Check the GPU

In Colab, select a GPU runtime before running these commands. `nvidia-smi` shows the attached GPU and `nvcc` is the CUDA compiler. CUDA C++ source files normally use the `.cu` extension.

In [ ]:
!nvidia-smi
!nvcc --version

## Host and device

A CUDA program normally has two sides:

```text
CPU = host
GPU = device
```

The CPU starts the program, prepares work, and launches kernels. The GPU runs the kernel code in parallel. CUDA does not move an entire program onto the GPU.

## A minimal CUDA program

`main()` is normal C++ host code. `__global__` marks a function as a kernel that runs on the GPU and can be launched from the host. Device-side `printf()` is useful for debugging, while `std::cout` remains host-side C++.

In [ ]:
%%writefile hello.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void hello_gpu() {
    printf("Hello from GPU!\n");
}

int main() {
    std::cout << "Hello from CPU!\n";

    hello_gpu<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}

In [ ]:
!nvcc hello.cu -o hello
!./hello

## Kernel launch and synchronization

The launch syntax below is CUDA-specific:

```cpp
hello_gpu<<<1, 1>>>();
```

It asks the CPU to launch one GPU block containing one GPU thread. Kernel launches are normally asynchronous with respect to the CPU. `cudaDeviceSynchronize()` makes the CPU wait until previously submitted GPU work finishes, which is useful here so the program does not exit before the kernel output appears.

## What this note establishes

- `.cu` files are compiled with `nvcc`
- `main()` runs on the host CPU
- `__global__` functions run as GPU kernels
- the host launches kernels with `<<< >>>`
- CUDA work is normally asynchronous until synchronization is requested